In [1]:
!pip install transformers datasets peft accelerate bitsandbytes -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 14.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 76.1/76.1 MB 10.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 116.3/116.3 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.5/143.5 kB 13.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 102.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import pandas as pd
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from datasets import Dataset
from sklearn.model_selection import train_test_split
import Token_use

from google.colab import drive
drive.mount('/content/drive')

# 1. Load dữ liệu CSV
# Đường dẫn đến file bạn đã tải lên
csv_path = "/content/drive/MyDrive/merged_data_(t-1).csv"
df = pd.read_csv(csv_path)
df["Closing Price_t+1"] = df["Closing Price"].shift(-1)
df["Closing Price_t+2"] = df["Closing Price"].shift(-2)
df["Closing Price_t+3"] = df["Closing Price"].shift(-3)

df = df.dropna()

# 2. Tiền xử lý: ghép tất cả cột đặc trưng thành chuỗi để tạo prompt
def row_to_prompt(row):
    # Chuyển điểm cảm xúc thành mô tả
    sentiment_map = {
        0: "tiêu cực",
        1: "trung lập",
        2: "tích cực"
    }
    sentiment_desc = sentiment_map.get(row["sentiment_score_lag1"], "không xác định")

    # Danh sách các đặc trưng được dùng
    features = [
        'Market Cap_lag1', 'Closing Price_lag1', 'Remaining Room %_lag1',
        'Remaining Room Shares_lag1', 'Total Sell Value_lag1',
        'Matched Sell Value_lag1', 'Matched Buy Value_lag1',
        'Negotiated Sell Value_lag1', 'Negotiated Sell Volume_lag1',
        'Negotiated Buy Value_lag1', 'Negotiated Buy Volume_lag1',
        'MA5', 'MA10'
    ]

    # Tạo phần mô tả input
    prompt = "Dự báo giá cổ phiếu trong 3 ngày tiếp theo dựa trên các thông tin sau:\n"
    for feature in features:
        prompt += f"- {feature}: {row[feature]}\n"
    prompt += f"- Tâm lý thị trường hôm nay: {sentiment_desc}\n\n"

    # Thêm phần câu hỏi & trả lời
    prompt += (
        "Câu hỏi: Dự đoán giá đóng cửa của cổ phiếu trong 3 ngày tới là bao nhiêu?\n"
        f"Trả lời: Ngày 1: {row['Closing Price_t+1']} VND, "
        f"Ngày 2: {row['Closing Price_t+2']} VND, "
        f"Ngày 3: {row['Closing Price_t+3']} VND"
    )

    return prompt


df = df.dropna()  # loại bỏ dòng chứa NaN
df["text"] = df.apply(row_to_prompt, axis=1)

# 3. Tạo Hugging Face Dataset
dataset = Dataset.from_pandas(df[["text"]])

# 4. Load tokenizer và model
model_name = "meta-llama/Llama-2-7b-chat-hf"
token = Token_use.token

tokenizer = AutoTokenizer.from_pretrained(model_name, use_auth_token=token)
tokenizer.pad_token = tokenizer.eos_token
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    use_auth_token=token,
    load_in_4bit=True,
    device_map="auto"
)

# 5. Cấu hình LoRA
model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.1,
    bias="none",
    task_type="CAUSAL_LM"
)
model = get_peft_model(model, lora_config)

# 6. Tokenize dữ liệu
def tokenize_function(example):
    tokens = tokenizer(example["text"], truncation=True, padding="max_length", max_length=512)
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function)
tokenized_dataset = tokenized_dataset.train_test_split(test_size=0.1)

# 7. TrainingArguments
training_args = TrainingArguments(
    output_dir="./output",
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    num_train_epochs=3,
    learning_rate=2e-4,
    logging_dir="./logs",
    save_strategy="epoch",
    fp16=True,
    remove_unused_columns=False
)

# 8. Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["test"]
)

# 9. Train
trainer.train()

# 10. Lưu mô hình fine-tuned
model.save_pretrained("./fine_tuned_llm_stock")
tokenizer.save_pretrained("./fine_tuned_llm_stock")


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


/usr/local/lib/python3.11/dist-packages/transformers/models/auto/tokenization_auto.py:898: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
/usr/local/lib/python3.11/dist-packages/transformers/models/auto/auto_factory.py:476: FutureWarning: The `use_auth_token` argument is deprecated and will be removed in v5 of Transformers. Please use `token` instead.
  warnings.warn(
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.


Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

Map:   0%|          | 0/227 [00:00<?, ? examples/s]

No label_names provided for model class `PeftModelForCausalLM`. Since `PeftModel` hides base models input arguments, if label_names is not given, label_names can't be set automatically within `Trainer`. Note that empty label_names list will be used instead.
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recom

Step,Training Loss
500,0.266200


/usr/local/lib/python3.11/dist-packages/torch/_dynamo/eval_frame.py:745: UserWarning: torch.utils.checkpoint: the use_reentrant parameter should be passed explicitly. In version 2.5 we will raise an exception if use_reentrant is not passed. use_reentrant=False is recommended, but if you need to preserve the current default behavior, you can pass use_reentrant=True. Refer to docs for more details on the differences between the two variants.
  return fn(*args, **kwargs)
/usr/local/lib/python3.11/dist-packages/peft/utils/other.py:1110: UserWarning: Unable to fetch remote file due to the following error 401 Client Error. (Request ID: Root=1-681cd2f6-7d7832ed593f7f6a569a93fa;ef042653-7c12-4be8-9274-b66a297b8324)

Cannot access gated repo for url https://huggingface.co/meta-llama/Llama-2-7b-chat-hf/resolve/main/config.json.
Access to model meta-llama/Llama-2-7b-chat-hf is restricted. You must have access to it and be authenticated to access it. Please log in. - silently ignoring the lookup f

('./fine_tuned_llm_stock/tokenizer_config.json',
 './fine_tuned_llm_stock/special_tokens_map.json',
 './fine_tuned_llm_stock/tokenizer.model',
 './fine_tuned_llm_stock/added_tokens.json',
 './fine_tuned_llm_stock/tokenizer.json')

In [9]:
results = trainer.evaluate()
print(results)

{'eval_loss': 0.20167632400989532, 'eval_runtime': 18.3413, 'eval_samples_per_second': 1.254, 'eval_steps_per_second': 1.254, 'epoch': 3.0}


In [10]:
def make_inference_prompt(row):
    sentiment_map = {0: "tiêu cực", 1: "trung lập", 2: "tích cực"}
    sentiment_desc = sentiment_map.get(row["sentiment_score_lag1"], "không xác định")

    features = [
        'Market Cap_lag1', 'Closing Price_lag1', 'Remaining Room %_lag1',
        'Remaining Room Shares_lag1', 'Total Sell Value_lag1',
        'Matched Sell Value_lag1', 'Matched Buy Value_lag1',
        'Negotiated Sell Value_lag1', 'Negotiated Sell Volume_lag1',
        'Negotiated Buy Value_lag1', 'Negotiated Buy Volume_lag1',
        'MA5', 'MA10'
    ]

    prompt = "Dự báo giá cổ phiếu trong 3 ngày tiếp theo dựa trên các thông tin sau:\n"
    for f in features:
        prompt += f"- {f}: {row[f]}\n"
    prompt += f"- Tâm lý thị trường hôm nay: {sentiment_desc}\n\n"
    prompt += "Câu hỏi: Dự đoán giá đóng cửa của cổ phiếu trong 3 ngày tới là bao nhiêu?\nTrả lời:"
    return prompt


In [16]:
# Dữ liệu ngày 13/3/2025 , Dự đoán 3 ngày kế tiếp
sample_input = {
    'Market Cap_lag1': 202272013,
    'Closing Price_lag1': 137500.0,
    'Remaining Room %_lag1': 4.59,
    'Remaining Room Shares_lag1': 67582207,
    'Total Sell Value_lag1': 324361,
    'Matched Sell Value_lag1': 272010,
    'Matched Buy Value_lag1': 194553,
    'Negotiated Sell Value_lag1': 52321,
    'Negotiated Sell Volume_lag1': 380000,
    'Negotiated Buy Value_lag1': 52321,
    'Negotiated Buy Volume_lag1': 380000,
    'MA5': 139780.0,
    'MA10': 140280.0,
    'sentiment_score_lag1': 1
}

In [17]:
input_prompt = make_inference_prompt(sample_input)

inputs = tokenizer(input_prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**inputs, max_new_tokens=50)

result = tokenizer.decode(outputs[0], skip_special_tokens=True)
print(result)


Dự báo giá cổ phiếu trong 3 ngày tiếp theo dựa trên các thông tin sau:
- Market Cap_lag1: 202272013
- Closing Price_lag1: 137500.0
- Remaining Room %_lag1: 4.59
- Remaining Room Shares_lag1: 67582207
- Total Sell Value_lag1: 324361
- Matched Sell Value_lag1: 272010
- Matched Buy Value_lag1: 194553
- Negotiated Sell Value_lag1: 52321
- Negotiated Sell Volume_lag1: 380000
- Negotiated Buy Value_lag1: 52321
- Negotiated Buy Volume_lag1: 380000
- MA5: 139780.0
- MA10: 140280.0
- Tâm lý thị trường hôm nay: trung lập

Câu hỏi: Dự đoán giá đóng cửa của cổ phiếu trong 3 ngày tới là bao nhiêu?
Trả lời: Ngày 1: 138000.0 VND, Ngày 2: 138500.0 VND, Ngày 3: 138000.
